In [4]:
import geopandas as gpd, pyogrio
import pandas as pd
from shapely.validation import make_valid

# ---- paths ----
PTS_PATH  = "matilda_final_fixed.gpkg"
PTS_LAYER = "matilda_final_fixed_3308"
LITH_PATH = "clipped_lithology.gpkg"
LITH_LAYER= None                      # auto-pick first if None
AOI_PATH  = "multiparts.gpkg"
AOI_LAYER = "multiparts"
TARGET_CRS = "EPSG:3308"

def fix_valid(g):
    try:
        return make_valid(g)          # shapely >= 2
    except Exception:
        try:
            return g if g.is_valid else g.buffer(0)
        except Exception:
            return g

def first_layer(path):
    # pyogrio returns: (name, geometry_type, feature_count, fields, metadata)
    return pyogrio.list_layers(path)[0][0]

if LITH_LAYER is None:
    LITH_LAYER = first_layer(LITH_PATH)

# ---- read with pyogrio (no Fiona) ----
pts  = pyogrio.read_dataframe(PTS_PATH,  layer=PTS_LAYER)
aoi  = pyogrio.read_dataframe(AOI_PATH,  layer=AOI_LAYER)
lith = pyogrio.read_dataframe(LITH_PATH, layer=LITH_LAYER)

# ---- CRS handling ----
if pts.crs is None:
    raise ValueError("Points layer has no CRS; please assign it before running.")
if lith.crs is None:
    # your clipped file should already be projected, but set to WGS84 if missing, then reproject
    lith = lith.set_crs("EPSG:4326")

pts  = pts.to_crs(TARGET_CRS)
aoi  = aoi.to_crs(TARGET_CRS)
lith = lith.to_crs(TARGET_CRS)

# clean invalid/empty polygons in lith and AOI
lith["geometry"] = lith.geometry.map(fix_valid)
lith = lith[~lith.is_empty & lith.geometry.notna()].copy()
aoi["geometry"]  = aoi.geometry.map(fix_valid)
aoi  = aoi[~aoi.is_empty & aoi.geometry.notna()].copy()

print("CRS:", pts.crs, lith.crs, aoi.crs)
print("Counts  points:", len(pts), "  lith polys:", len(lith), "  AOI parts:", len(aoi))

# ---- points inside AOI (NO unary_union) ----
# Keep only points that spatially join with AOI (within/intersects)
pts_idx_in = gpd.sjoin(pts[["geometry"]], aoi[["geometry"]], how="inner", predicate="within").index.unique()
pts_in = pts.loc[pts.index.isin(pts_idx_in)].copy()
print("Points inside AOI:", len(pts_in), " (outside:", len(pts) - len(pts_in), ")")

# ---- pick a lithology attribute ----
preferred = ["UNIT_NAME","UNITNAME","ROCK_NAME","GEOL_NAME","UNIT_CODE","GEOL_CODE","LITHOLOGY","LITH"]
colsU = {c.upper(): c for c in lith.columns if c != "geometry"}
lith_field = None
for c in preferred:
    if c in colsU:
        lith_field = colsU[c]; break
if lith_field is None:
    for c in lith.columns:
        if c != "geometry" and lith[c].dtype == object:
            lith_field = c; break
print("Chosen lithology field:", lith_field)

# ---- join lithology to points (intersects is fine for field mapping) ----
pts_join = gpd.sjoin(pts_in, lith[[lith_field, "geometry"]], how="left", predicate="intersects")
miss = pts_join[lith_field].isna().sum()
print("Points without lithology after join:", miss)
print("Top lith classes at points:\n", pts_join[lith_field].value_counts().head(15))

# (optional) write a quick preview file
# pts_join[['activity', lith_field, 'geometry']].to_file('pts_with_lith.gpkg', layer='pts_with_lith', driver='GPKG')


CRS: EPSG:3308 EPSG:3308 EPSG:3308
Counts  points: 1109   lith polys: 143   AOI parts: 1
Points inside AOI: 1103  (outside: 6 )
Chosen lithology field: unit_name
Points without lithology after join: 0
Top lith classes at points:
 unit_name
Hawkesbury Sandstone                                       441
Ashfield Shale                                             227
Coastal deposits - dune facies                             227
Anthropogenic deposits - reclaimed estuarine areas          83
Anthropogenic deposits                                      72
Coastal deposits - bedrock-mantling dune facies             25
Coastal deposits - beach facies                             20
Alluvial fan deposits                                        5
Anthropogenic stored water, pondage, reservoirs, canals      2
Estuarine basin and bay (subaqueous)                         1
Name: count, dtype: int64


In [7]:
import numpy as np, pandas as pd
import geopandas as gpd, pyogrio
from shapely.validation import make_valid
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from pykrige.rk import RegressionKriging
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize

# ----------------- config -----------------
PTS_PATH   = "matilda_final_fixed.gpkg"
PTS_LAYER  = "matilda_final_fixed_3308"
AOI_PATH   = "multiparts.gpkg"
AOI_LAYER  = "multiparts"
LITH_PATH  = "clipped_lithology.gpkg"
LITH_LAYER = None               # auto-pick first if None
ACTIVITY   = "activity"
LITH_ATTR  = "unit_name"        # from your Step 1 output
CRS        = "EPSG:3308"

# grid target size (approximate)
NX = 1000
NY = 1000

# model / variogram settings
ALPHA_RIDGE = 1.0
VARIO_MODEL = "spherical"
# rule-of-thumb initial variogram params = [sill, range, nugget]; we compute below
# ----------------------------------------

def fix_valid(g):
    try:
        return make_valid(g)
    except Exception:
        try:
            return g if g.is_valid else g.buffer(0)
        except Exception:
            return g

def first_layer(path):
    return pyogrio.list_layers(path)[0][0]

# ---- load data
pts  = pyogrio.read_dataframe(PTS_PATH,  layer=PTS_LAYER).to_crs(CRS)
aoi  = pyogrio.read_dataframe(AOI_PATH,  layer=AOI_LAYER).to_crs(CRS)
if LITH_LAYER is None:
    LITH_LAYER = first_layer(LITH_PATH)
lith = pyogrio.read_dataframe(LITH_PATH, layer=LITH_LAYER)
if lith.crs is None:
    # your file should already be projected; if not, assume WGS84 then reproject
    lith = lith.set_crs("EPSG:4326")
lith = lith.to_crs(CRS)

# clean invalid geoms
lith["geometry"] = lith.geometry.map(fix_valid)
lith = lith[~lith.is_empty & lith.geometry.notna()].copy()
aoi["geometry"]  = aoi.geometry.map(fix_valid)
aoi  = aoi[~aoi.is_empty & aoi.geometry.notna()].copy()

# ---- keep only points inside AOI
pts = pts[pts.geometry.notna()]
pts = pts.dropna(subset=[ACTIVITY]).copy()
pts_idx_in = gpd.sjoin(pts[["geometry"]], aoi[["geometry"]], how="inner", predicate="within").index.unique()
pts_in = pts.loc[pts.index.isin(pts_idx_in)].copy()

# ---- attach lithology to points (for drift)
pts_in = gpd.sjoin(pts_in, lith[[LITH_ATTR, "geometry"]], how="left", predicate="intersects")
if pts_in[LITH_ATTR].isna().any():
    pts_in[LITH_ATTR] = pts_in[LITH_ATTR].fillna("UNKNOWN")

# =========================
# Step 2 — Build the grid
# =========================
xmin, ymin, xmax, ymax = aoi.total_bounds
width  = xmax - xmin
height = ymax - ymin
res_x = width  / NX
res_y = height / NY
transform = from_origin(xmin, ymax, res_x, res_y)

# AOI mask raster (1 inside AOI, 0 outside)
aoi_shapes = [(geom, 1) for geom in aoi.geometry]
mask_r = rasterize(
    aoi_shapes, out_shape=(NY, NX), transform=transform,
    fill=0, dtype="uint8", all_touched=True
)

# Lithology categorical raster (integer codes)
# Map unit_name -> code (1..K), 0 = outside/not mapped
lith = lith.dropna(subset=[LITH_ATTR]).copy()
names = pd.unique(lith[LITH_ATTR])
names_sorted = np.sort(names.astype(str))  # stable order
name2code = {n: i+1 for i, n in enumerate(names_sorted)}
code2name = np.array(["<OUT>"] + list(names_sorted), dtype=object)

lith_shapes = ((geom, name2code.get(str(row[LITH_ATTR]), 0))
               for _, row in lith.iterrows()
               for geom in ([row.geometry] if row.geometry.geom_type != "MultiPolygon"
                            else list(row.geometry.geoms)))
cat_r = rasterize(
    lith_shapes, out_shape=(NY, NX), transform=transform,
    fill=0, dtype="int32", all_touched=True
)

# indices of prediction locations (inside AOI)
rows, cols = np.where(mask_r == 1)
# coordinates of cell centers
x_coords = xmin + (cols + 0.5) * res_x
y_coords = ymax - (rows + 0.5) * res_y
grid_coords = np.column_stack([x_coords, y_coords])

# grid categories (strings) for drift (align to code2name)
grid_codes = cat_r[rows, cols]
grid_lith_names = code2name[grid_codes]  # object array of names

# ----------------------------
# Predict in batches (manual variance handling for older PyKrige)
# ----------------------------

def batched(total, batch_size=120_000):
    start = 0
    while start < total:
        end = min(start + batch_size, total)
        yield start, end
        start = end

N = grid_coords.shape[0]
pred = np.full(N, np.nan, dtype="float64")
var  = np.full(N, np.nan, dtype="float64")

for s, e in batched(N):
    # 1) Drift (regression) part
    cats = grid_lith_names[s:e].astype(str).reshape(-1, 1)
    Xg = ohe.transform(cats)                                  # one-hot using the same encoder
    drift_pred = rk.regression_model.predict(Xg)              # Ridge predictions

    # 2) Residual kriging part (pred + variance)
    #    Use the underlying OK/UK model directly.
    #    execute(style="points", xpoints, ypoints [, backend=...]) -> (z, ss)
    x = grid_coords[s:e, 0]
    y = grid_coords[s:e, 1]
    z_res, ss_res = rk.krige.model.execute("points", x, y)    # variance = ss_res

    # 3) Combine
    pred[s:e] = drift_pred + z_res
    var[s:e]  = ss_res

# Back to full rasters
pred_img = np.full((NY, NX), np.nan, dtype="float32")
var_img  = np.full((NY, NX), np.nan, dtype="float32")
pred_img[rows, cols] = pred.astype("float32")
var_img[rows,  cols] = var.astype("float32")

# Write GeoTIFFs (same as before)
profile = {
    "driver": "GTiff", "height": NY, "width": NX, "count": 1, "dtype": "float32",
    "crs": CRS, "transform": transform, "compress": "deflate"
}
with rasterio.open("rk_pred.tif", "w", **profile) as dst:
    dst.write(pred_img, 1)
with rasterio.open("rk_var.tif", "w", **profile) as dst:
    dst.write(var_img, 1)

print("Done. Wrote rk_pred.tif and rk_var.tif")

/opt/homebrew/Caskroom/miniconda/base/envs/geo-rk/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/geo-rk/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/geo-rk/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/geo-rk/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/homebrew/Caskroom

Done. Wrote rk_pred.tif and rk_var.tif


In [8]:
import rasterio
import numpy as np

for fn in ["rk_pred.tif","rk_var.tif"]:
    with rasterio.open(fn) as src:
        arr = src.read(1)
        arr = arr[~np.isnan(arr)]
        print(fn, "→ min, mean, max:", arr.min(), arr.mean(), arr.max())

rk_pred.tif → min, mean, max: 1.4991884 4.982946 8.349509
rk_var.tif → min, mean, max: 0.09732288 0.3397968 0.8887213


In [ ]:
import numpy as np
import geopandas as gpd
from shapely.validation import make_valid
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from pykrige.rk import RegressionKriging

# Paths and parameters
PTS_PATH   = "matilda_final_fixed.gpkg"
PTS_LAYER  = "matilda_final_fixed_3308"
LITH_PATH  = "clipped_lithology.gpkg"
LITH_LAYER = "geologyrock_units_nsw"
AOI_PATH   = "multiparts.gpkg"
AOI_LAYER  = "multiparts"
LITH_ATTR  = "unit_name"
CRS        = "EPSG:3308"
N_SPLITS   = 5
SEED       = 42

# Helper: fix invalid geometries
def fix_valid(g):
    try: 
        from shapely.validation import make_valid
        return make_valid(g)
    except:
        return g if g.is_valid else g.buffer(0)

# 1) Load data
pts = gpd.read_file(PTS_PATH, layer=PTS_LAYER).to_crs(CRS)
pts = pts.dropna(subset=["activity"])
aoi = gpd.read_file(AOI_PATH, layer=AOI_LAYER).to_crs(CRS)
lith = gpd.read_file(LITH_PATH, layer=LITH_LAYER).to_crs(CRS)
lith["geometry"] = lith.geometry.map(fix_valid)
lith = lith[~lith.is_empty]

# 2) Keep only points inside your AOI
pts = gpd.sjoin(pts, aoi[["geometry"]], how="inner", predicate="within").drop(columns="index_right")

# 3) Attach lithology
pts = gpd.sjoin(pts, lith[[LITH_ATTR,"geometry"]], how="left", predicate="intersects")
pts[LITH_ATTR] = pts[LITH_ATTR].fillna("UNKNOWN")

# 4) Build design matrix and coordinates
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
X = ohe.fit_transform(pts[[LITH_ATTR]].astype(str))
y = pts["activity"].values
coords = np.vstack([pts.geometry.x, pts.geometry.y]).T

# Precompute domain size for variogram range
xmin, ymin, xmax, ymax = aoi.total_bounds
max_dim = max(xmax-xmin, ymax-ymin)

# 5) Cross‐validation
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
rmses, r2s = [], []

for train_idx, test_idx in kf.split(X):
    # Regression on train
    reg = Ridge(alpha=1.0).fit(X[train_idx], y[train_idx])
    resid = y[train_idx] - reg.predict(X[train_idx])
    # Variogram params
    sill = np.var(resid, ddof=1)
    vrange = 0.3 * max_dim
    nugget = 0.1 * sill
    vario_params = [sill, vrange, nugget]
    # Fit RK on train
    rk = RegressionKriging(
        regression_model=Ridge(alpha=1.0),
        n_closest_points=24,
        variogram_model="spherical",
        variogram_parameters=vario_params
    )
    rk.fit(X[train_idx], coords[train_idx], y[train_idx])
    # Predict on test
    drift = rk.regression_model.predict(X[test_idx])
    xt, yt = coords[test_idx,0], coords[test_idx,1]
    zres, varres = rk.krige.model.execute("points", xt, yt)
    ypred = drift + zres
    # Metrics
    rmses.append(np.sqrt(mean_squared_error(y[test_idx], ypred)))
    r2s.append(r2_score(y[test_idx], ypred))

rmses = np.array(rmses); r2s = np.array(r2s)
print(f"CV RMSE: {rmses.mean():.3f} ± {rmses.std():.3f}")
print(f"CV R²:   {r2s.mean():.3f} ± {r2s.std():.3f}")


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'